# Broadcasting experiments

Configure and run broadcasting hardware, exact and sampled simulations, convergence, and the encoded or bare memory benchmark here. Every execution switch is **False** by default. Run all cells to inspect settings and saved job status without account access, submission, simulations, or file writes.

Every result is a self-contained JSON file directly in `results/`. A hardware job file includes its configuration, frozen circuits, submission attempt, receipt, and all measured cases. Completed measurements are immutable. Use `visualizations.ipynb` for inspection, plots, and manuscript exports.


In [1]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "broadcasting").is_dir():
    raise RuntimeError("Open this notebook from the Broadcasting project root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from broadcasting import ProtocolConfig, ExactBackend, SamplingBackend
from broadcasting.results import save_run, load_run
from broadcasting.experiments import (
    make_experiment_config, make_memory_config, plan_experiment,
    prepare_experiment, load_experiment, find_experiments, submit_experiment,
    collect_experiment, experiment_status, attach_job,
    build_memory_circuits, run_memory_reference,
)
from broadcasting.convergence import load_convergence, collect_convergence_repeats

RESULTS_DIR = ROOT / "results"


## Broadcasting hardware

The saved IBM profile contains account credentials. Choose an explicit backend. Existing matching job files can be resumed; change `experiment_id` to start another set with the same physical settings. Preparation archives circuits without submitting. Submission records the attempt before contacting Runtime and stores the job ID immediately. Collection finalizes each job file atomically.


In [2]:
IBM_PROFILE = "mprest1"
IBM_BACKEND = "ibm_kingston"
EXPERIMENT_SEED = 20260908
HARDWARE_REPEATS = 3
SCALING_EXPERIMENT_ID = "broadcasting-scaling-01"
DELAY_EXPERIMENT_ID = "broadcasting-delay-01"


### Zero-delay scaling

The matrix M=1,2,3 and N=1,2,3,4 uses 8,192 shots per case and three repeats at optimization level 3. Each repeat is a job containing all 12 cases in seeded interleaved order. Sender phases vary across repeats and share prefixes across receiver counts.


In [3]:
SCALING_SENDERS = [1, 2, 3]
SCALING_RECEIVERS = [1, 2, 3, 4]
scaling_config = make_experiment_config(
    "scaling", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id=SCALING_EXPERIMENT_ID, repeats=HARDWARE_REPEATS,
    seed=EXPERIMENT_SEED, sender_counts=SCALING_SENDERS,
    receiver_counts=SCALING_RECEIVERS,
)
assert scaling_config["shots"] == 8192
assert scaling_config["optimization_level"] == 3


### Receiver delay sweeps

M=1,N=2 uses 10,000 shots at each of 121 delays, 0–6000 dt in steps of 50 dt. Three repeats use distinct seeded sender phases. The archived device dt defines physical time; preparation rejects unsupported durations.


In [4]:
TAU_VALUES_DT = list(range(0, 6001, 50))
delay_config = make_experiment_config(
    "delay", runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id=DELAY_EXPERIMENT_ID, repeats=HARDWARE_REPEATS,
    seed=EXPERIMENT_SEED + 1000, tau_values_dt=TAU_VALUES_DT,
)
assert delay_config["shots"] == 10000
assert delay_config["optimization_level"] == 3
experiments = {
    "scaling": (scaling_config, find_experiments(scaling_config, results_dir=RESULTS_DIR)),
    "delay": (delay_config, find_experiments(delay_config, results_dir=RESULTS_DIR)),
}


### Offline budget and phase preview

Review the full shot budget and phase samples before enabling preparation. Submitted circuit order is preserved, but does not specify the provider's chronological execution order.


In [5]:
plans = {name: plan_experiment(config) for name, (config, _) in experiments.items()}
for name, plan in plans.items():
    print(f"\n{name.upper()} → {experiments[name][1]}")
    print(json.dumps({key: value for key, value in plan.items() if key != "repeats"}, indent=2))
    for repeat in plan["repeats"]:
        print(f"Repeat {repeat['repeat_index']} phase seed {repeat['phases']['phase_seed']}:")
        for case, samples in zip(plan["cases"], repeat["phases"]["theta_samples_by_case"]):
            print(f"  {case['id']}: {samples}")
print(f"\nCOMBINED: {sum(plan['jobs'] for plan in plans.values())} jobs; "
      f"{sum(plan['total_shots'] for plan in plans.values()):,} shots")



SCALING → []
{
  "experiment_id": "broadcasting-scaling-01",
  "backend": "ibm_kingston",
  "runtime_account": "mprest1",
  "config_sha256": "8f65bc293ee20b5ce20ce811d7a935f96e5c21b366315b5b04334e4232f3a2a8",
  "jobs": 3,
  "pubs_per_job": 12,
  "total_pubs": 36,
  "total_shots": 294912,
  "shots_per_pub": 8192,
  "optimization_level": 3,
  "cases": [
    {
      "id": "m1_n1",
      "M": 1,
      "N": 1,
      "logical_qubits": 2,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n2",
      "M": 1,
      "N": 2,
      "logical_qubits": 4,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n3",
      "M": 1,
      "N": 3,
      "logical_qubits": 5,
      "shots_per_repeat": 8192
    },
    {
      "id": "m1_n4",
      "M": 1,
      "N": 4,
      "logical_qubits": 7,
      "shots_per_repeat": 8192
    },
    {
      "id": "m2_n1",
      "M": 2,
      "N": 1,
      "logical_qubits": 3,
      "shots_per_repeat": 8192
    },
    {
      "id": "m2_n2",
      "M": 2,
    

### Prepare, submit, and collect

Enable each action explicitly. Matching job files are reused, and each selected job's configuration is checked before submission. If an attempt is ambiguous after an interruption, use the recovery cell to attach its matching tagged IBM job ID. Submission never retries an ambiguous attempt.


In [6]:
PREPARE_SCALING = False
PREPARE_DELAY = False

for name, enabled in {"scaling": PREPARE_SCALING, "delay": PREPARE_DELAY}.items():
    if enabled:
        config, job_files = experiments[name]
        if job_files:
            for path in job_files:
                load_experiment(path, expected_config=config)
        else:
            job_files = prepare_experiment(config, results_dir=RESULTS_DIR)
            experiments[name] = (config, job_files)
        print(f"{name}: prepared job files", job_files)


In [7]:
SUBMIT_SCALING = False

if SUBMIT_SCALING:
    config, job_files = experiments["scaling"]
    if not job_files:
        raise ValueError("Prepare the scaling experiment first.")
    for path in job_files:
        load_experiment(path, expected_config=config)
    print("New scaling jobs:", submit_experiment(job_files))


In [8]:
SUBMIT_DELAY = False

if SUBMIT_DELAY:
    config, job_files = experiments["delay"]
    if not job_files:
        raise ValueError("Prepare the delay experiment first.")
    for path in job_files:
        load_experiment(path, expected_config=config)
    print("New delay jobs:", submit_experiment(job_files))


In [9]:
COLLECT_SCALING = False
COLLECT_DELAY = False

for name, enabled in {"scaling": COLLECT_SCALING, "delay": COLLECT_DELAY}.items():
    if enabled:
        config, job_files = experiments[name]
        for path in job_files:
            load_experiment(path, expected_config=config)
        print(f"{name}: collected", collect_experiment(job_files))
        print(json.dumps(experiment_status(job_files), indent=2))


In [10]:
for name, (_, job_files) in experiments.items():
    print(name, json.dumps(experiment_status(job_files), indent=2))


scaling {
  "jobs": []
}


delay {
  "jobs": [
    {
      "path": "/home/matt/Projects/Qiskit Projects/Broadcasting/results/job_5bfbc6524097416e8c6642cc0568c656_repeat_000.json",
      "experiment_id": "broadcasting-delay-01",
      "repeat_index": 0,
      "state": "collected",
      "job_id": "dago8l8mhr3c73e5ejq0",
      "measurements": 1
    },
    {
      "path": "/home/matt/Projects/Qiskit Projects/Broadcasting/results/job_5bfbc6524097416e8c6642cc0568c656_repeat_001.json",
      "experiment_id": "broadcasting-delay-01",
      "repeat_index": 1,
      "state": "collected",
      "job_id": "dago8lomhr3c73e5ejrg",
      "measurements": 1
    },
    {
      "path": "/home/matt/Projects/Qiskit Projects/Broadcasting/results/job_5bfbc6524097416e8c6642cc0568c656_repeat_002.json",
      "experiment_id": "broadcasting-delay-01",
      "repeat_index": 2,
      "state": "collected",
      "job_id": "dago8m39k43c73adlll0",
      "measurements": 1
    }
  ]
}


In [11]:
ATTACH_JOB = False
RECOVERY_JOB_JSON = None  # Path to the attempted job JSON in results/.
RECOVERY_JOB_ID = ""

if ATTACH_JOB:
    if RECOVERY_JOB_JSON is None or not RECOVERY_JOB_ID:
        raise ValueError("Set the attempted job JSON and the matching tagged IBM job ID.")
    print(attach_job(RECOVERY_JOB_JSON, RECOVERY_JOB_ID))


## Exact and sampled broadcasting

Both methods save the common measurement format. Choose the sender/receiver sizes, state, noise grid, branch selection, and seed below. Sampling requires receiver QEC; an exact encoded run can require substantial local memory.


In [12]:
SIMULATION_CONFIG = ProtocolConfig(
    M=1, N=2, alpha=1 / np.sqrt(2), thetas=[0.0],
    p_list=np.linspace(0, 1, 21).tolist(), use_qec=False,
    outcomes_list=[0], seed=42,
)
SAMPLED_CONFIG = replace(SIMULATION_CONFIG, use_qec=True, n_samples=200)
print("Exact configuration:", SIMULATION_CONFIG)
print("Sampled configuration:", SAMPLED_CONFIG)


Exact configuration: ProtocolConfig(M=1, N=2, alpha=np.float64(0.7071067811865475), thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=False, outcomes_list=[0], tau=None, n_samples=None, seed=42, linear_feedforward=True)
Sampled configuration: ProtocolConfig(M=1, N=2, alpha=np.float64(0.7071067811865475), thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=True, outcomes_list=[0], tau=None, n_samples=200, seed=42, linear_feedforward=True)


In [13]:
RUN_EXACT = False
RUN_SAMPLED = False

if RUN_EXACT:
    exact_result = ExactBackend().run(SIMULATION_CONFIG)
    print("Saved exact result:", save_run(exact_result, SIMULATION_CONFIG, results_dir=RESULTS_DIR))
if RUN_SAMPLED:
    sampled_result = SamplingBackend().run(SAMPLED_CONFIG)
    print("Saved sampled result:", save_run(sampled_result, SAMPLED_CONFIG, results_dir=RESULTS_DIR))


## Monte Carlo-to-exact convergence

Inspect the saved convergence configuration and trajectory grid, then enable independent seed repetitions when needed. Results retain the exact reference and per-seed measurements; this notebook prints summaries only.


In [14]:
convergence_study = load_convergence(results_dir=RESULTS_DIR)
print("Convergence configuration:", convergence_study.config)
print("Trajectory counts:", convergence_study.sample_counts)
print("Saved repetitions:", len(convergence_study.repetitions))


Convergence configuration: ProtocolConfig(M=1, N=2, alpha=0.7071067811865475, thetas=[0.0], p_list=[0.0, 0.05, 0.1, 0.15000000000000002, 0.2, 0.25, 0.30000000000000004, 0.35000000000000003, 0.4, 0.45, 0.5, 0.55, 0.6000000000000001, 0.65, 0.7000000000000001, 0.75, 0.8, 0.8500000000000001, 0.9, 0.9500000000000001, 1.0], use_qec=True, outcomes_list=[0], tau=None, n_samples=None, seed=0, linear_feedforward=True)
Trajectory counts: [50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000, 100000]
Saved repetitions: 4


In [15]:
RUN_CONVERGENCE = False
CONVERGENCE_SEEDS = [3, 4]
CONVERGENCE_STUDY_ID = "repeats_02"  # Reuse to resume this study; change for a new study.

if RUN_CONVERGENCE:
    collect_convergence_repeats(
        convergence_study, repeats=len(CONVERGENCE_SEEDS), seeds=CONVERGENCE_SEEDS,
        results_dir=RESULTS_DIR, study_id=CONVERGENCE_STUDY_ID,
    )
    convergence_study = load_convergence(results_dir=RESULTS_DIR)
    print("Saved repetitions:", len(convergence_study.repetitions))


## Encoded or bare memory benchmark

This single-qubit delay benchmark uses the same measured-result schema as broadcasting. Select QEC, state preparation, delays, shots, optimization level, and seeds below. Circuit construction, noise-free reference, hardware preparation, submission, and collection each have a separate switch. Hardware preparation produces self-contained job JSONs in `results/`; the recovery cell also accepts these memory job files.


In [16]:
MEMORY_USE_QEC = True
MEMORY_SEED = 42
memory_rng = np.random.default_rng(MEMORY_SEED)
MEMORY_THETA = float(memory_rng.uniform(0, np.pi))
MEMORY_PHI = float(memory_rng.uniform(0, 2 * np.pi))
memory_config = make_memory_config(
    runtime_account=IBM_PROFILE, backend=IBM_BACKEND,
    experiment_id="qec-memory-01", use_qec=MEMORY_USE_QEC,
    theta=MEMORY_THETA, phi=MEMORY_PHI,
    tau_values_dt=np.linspace(0, 6000, 21).astype(int).tolist(),
    shots=8192, optimization_level=0, seed=MEMORY_SEED,
)
print(json.dumps(plan_experiment(memory_config), indent=2))

memory_jobs = find_experiments(memory_config, results_dir=RESULTS_DIR)


{
  "experiment_id": "qec-memory-01",
  "backend": "ibm_kingston",
  "runtime_account": "mprest1",
  "config_sha256": "d299cc32c67b73fd57bac158afb676b9d47d5ab397618d499f8e095b4e9a7227",
  "jobs": 1,
  "pubs_per_job": 21,
  "total_pubs": 21,
  "total_shots": 172032,
  "shots_per_pub": 8192,
  "optimization_level": 0,
  "cases": [
    {
      "id": "memory",
      "circuit_qubits": 9
    }
  ],
  "order_note": "Submitted PUB order is archived; device chronological order is not guaranteed. See Runtime execution spans when available.",
  "repeats": [
    {
      "repeat_index": 0,
      "pub_order": [
        {
          "case_index": 0,
          "case_id": "memory",
          "canonical_index": 19,
          "tau_index": 19,
          "tau_dt": 5700
        },
        {
          "case_index": 0,
          "case_id": "memory",
          "canonical_index": 5,
          "tau_index": 5,
          "tau_dt": 1500
        },
        {
          "case_index": 0,
          "case_id": "memory",
 

In [17]:
BUILD_MEMORY = False
if BUILD_MEMORY:
    memory_circuit, memory_bound_circuits, memory_register = build_memory_circuits(memory_config)
    print(f"Memory circuit: {memory_circuit.num_qubits} qubits, depth {memory_circuit.depth()}; "
          f"{len(memory_bound_circuits)} delays; measurement register {memory_register}")


In [18]:
RUN_MEMORY_REFERENCE = False
if RUN_MEMORY_REFERENCE:
    memory_reference_path = run_memory_reference(memory_config, results_dir=RESULTS_DIR)
    memory_reference = load_run(memory_reference_path)
    print("Saved noise-free memory reference:", memory_reference_path)
    print("Mean fidelity:", float(np.mean(memory_reference["fidelities"])))


In [19]:
PREPARE_MEMORY = False
SUBMIT_MEMORY = False
COLLECT_MEMORY = False

if PREPARE_MEMORY:
    if memory_jobs:
        for path in memory_jobs:
            load_experiment(path, expected_config=memory_config)
    else:
        memory_jobs = prepare_experiment(memory_config, results_dir=RESULTS_DIR)
if SUBMIT_MEMORY:
    if not memory_jobs:
        raise ValueError("Prepare the memory experiment first.")
    for path in memory_jobs:
        load_experiment(path, expected_config=memory_config)
    print("Memory jobs:", submit_experiment(memory_jobs))
if COLLECT_MEMORY:
    for path in memory_jobs:
        load_experiment(path, expected_config=memory_config)
    print("Collected memory jobs:", collect_experiment(memory_jobs))
